# Feature Engineering (RAWG → dataset model-ready)

Este notebook:
- Extrae el dataset desde PostgreSQL (RDS) con la query de entrenamiento.
- Valida / recalcula la variable objetivo **is_success** con tu lógica (éxito = ≥2/4 criterios).
- Genera features derivadas adicionales (recency_score, engagement_score, etc.).
- Prepara un dataset **model-ready** y lo guarda en `data/processed/`.


## 0) Setup


In [7]:
# Si estás en EC2, instala dependencias si hace falta:
# !pip install -U pandas numpy sqlalchemy psycopg2-binary python-dotenv

import os
import numpy as np
import pandas as pd
from datetime import datetime

import sys
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Project root añadido a sys.path:", PROJECT_ROOT)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

CURRENT_YEAR = datetime.now().year
CURRENT_YEAR


Project root añadido a sys.path: C:\Users\crisr\dev\rawg-aws-ml-analytics


2026

## 1) Credenciales y conexión a RDS (sin exponer secretos)

Opción recomendada en EC2:
- variables de entorno: `DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER`, `DB_PASSWORD`
- opcional: `DB_SCHEMA` (por defecto `rawg`)


In [8]:
import os
import psycopg2
from sqlalchemy import create_engine
from utils.aws_secrets import get_rds_credentials

SECRET_NAME = os.getenv("DB_SECRET_NAME", "Postgre")
DB_SCHEMA = os.getenv("DB_SCHEMA", "rawg")

def get_engine():
    creds = get_rds_credentials(SECRET_NAME)

    engine = create_engine(
        f"postgresql+psycopg2://{creds['username']}:{creds['password']}"
        f"@{creds['host']}:{creds['port']}/{creds['dbname']}?sslmode=require",
        pool_pre_ping=True,      # evita conexiones muertas
        pool_size=5,             # ajustable
        max_overflow=10
    )

    return engine


## 2) Query: dataset base para entrenamiento

Nota: tu query ya calcula `is_success`, pero luego lo validamos con tu función `calculate_success`
para garantizar coherencia con la métrica de éxito (≥2 criterios).


In [12]:
SQL_TRAIN_DATASET = f"""
WITH game_stats AS (
  SELECT 
    g.game_id,
    g.game_name,

    g.game_rating,
    g.ratings_count,
    g.game_added,
    g.suggestions_count,
    g.playtime,

    COALESCE(gs.playing, 0) AS playing,
    COALESCE(gs.owned, 0) AS owned,
    COALESCE(gs.toplay, 0) AS toplay,
    COALESCE(gs.beaten, 0) AS beaten,
    COALESCE(gs.dropped, 0) AS dropped,

    COUNT(DISTINCT gp.platform_id) AS num_platforms,
    COUNT(DISTINCT gst.store_id) AS num_stores,
    COUNT(DISTINCT gg.genre_id) AS num_genres,
    COUNT(DISTINCT gt.tag_id) AS num_tags,

    COALESCE(e.esrb_name, 'Unknown') AS esrb_name,
    SUBSTRING(g.released_ym, 1, 4)::integer AS release_year,

    CASE WHEN EXISTS (
      SELECT 1 FROM {DB_SCHEMA}.game_tags gt2 
      JOIN {DB_SCHEMA}.tags t ON t.tag_id = gt2.tag_id
      WHERE gt2.game_id = g.game_id AND t.tag_name = 'Multiplayer'
    ) THEN 1 ELSE 0 END AS has_multiplayer,

    CASE WHEN EXISTS (
      SELECT 1 FROM {DB_SCHEMA}.game_tags gt2 
      JOIN {DB_SCHEMA}.tags t ON t.tag_id = gt2.tag_id
      WHERE gt2.game_id = g.game_id AND t.tag_name = 'Singleplayer'
    ) THEN 1 ELSE 0 END AS has_singleplayer,

    CASE WHEN EXISTS (
      SELECT 1 FROM {DB_SCHEMA}.game_genres gg2
      JOIN {DB_SCHEMA}.genres ge ON ge.genre_id = gg2.genre_id
      WHERE gg2.game_id = g.game_id AND ge.genre_name = 'Indie'
    ) THEN 1 ELSE 0 END AS is_indie

  FROM {DB_SCHEMA}.games g
  LEFT JOIN {DB_SCHEMA}.games_status gs ON g.game_id = gs.game_id
  LEFT JOIN {DB_SCHEMA}.esrb_ratings e ON g.esrb_id = e.esrb_id
  LEFT JOIN {DB_SCHEMA}.game_platforms gp ON g.game_id = gp.game_id
  LEFT JOIN {DB_SCHEMA}.game_stores gst ON g.game_id = gst.game_id
  LEFT JOIN {DB_SCHEMA}.game_genres gg ON g.game_id = gg.game_id
  LEFT JOIN {DB_SCHEMA}.game_tags gt ON g.game_id = gt.game_id

  WHERE g.game_rating IS NOT NULL
    AND g.released_ym IS NOT NULL

  GROUP BY 
    g.game_id, g.game_name, g.game_rating, g.ratings_count,
    g.game_added, g.suggestions_count, g.playtime, g.released_ym,
    gs.playing, gs.owned, gs.toplay, gs.beaten, gs.dropped,
    e.esrb_name
),

percentiles AS (
  SELECT
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY game_added) AS game_added_p75,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY playtime) AS playtime_p75,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY playing) AS playing_p75
  FROM game_stats
)

SELECT 
  gs.*,
  p.game_added_p75,
  p.playtime_p75,
  p.playing_p75,

  ROUND(
    (gs.game_rating / NULLIF(LOG(gs.ratings_count + 1), 0))::numeric, 
    4
  ) AS rating_popularity_ratio,

  gs.playtime * gs.playing AS engagement_score,

  (2026 - gs.release_year) AS years_since_release,

  ROUND(
    (gs.game_rating * LOG(gs.ratings_count + 1))::numeric, 
    4
  ) AS quality_confidence,

  CASE 
    WHEN (gs.game_rating >= 4.0 AND gs.ratings_count >= 100)
      OR (gs.game_added >= p.game_added_p75)
      OR (gs.playtime >= p.playtime_p75 AND gs.game_rating >= 3.5)
      OR (gs.playing >= p.playing_p75)
    THEN 1 
    ELSE 0 
  END AS is_success

FROM game_stats gs
CROSS JOIN percentiles p
ORDER BY gs.game_id;
""".format(DB_SCHEMA=DB_SCHEMA)


In [13]:
engine = get_engine()
df = pd.read_sql_query(SQL_TRAIN_DATASET, engine)
df.shape, df.head(3)


((137179, 29),
    game_id                game_name  game_rating  ratings_count  game_added  suggestions_count  playtime  playing  owned  toplay  beaten  \
 0       10    G Prime Into The Rain          0.0              5          69                266         1        0     63       1       3   
 1       11  Replay: VHS is not dead          0.0              4           9                395         0        0      6       0       3   
 2       13   MagNets: Fully Charged          0.0              2           6                156         0        0      4       0       2   
 
    dropped  num_platforms  num_stores  num_genres  num_tags     esrb_name  release_year  has_multiplayer  has_singleplayer  is_indie  \
 0        0              3           2           2        20      Everyone          2016                0                 1         1   
 1        0              4           3           2         4      Everyone          2016                0                 0         0   
 2      

## 3) Target: métrica de éxito (≥ 2 de 4 criterios)


In [14]:
def calculate_success(row):
    # === CRITERIO 1: Alta Calidad Validada ===    
    high_quality = (row['game_rating'] >= 4.0 and row['ratings_count'] >= 100)
    # === CRITERIO 2: Alta Popularidad ===    
    high_popularity = (row['game_added'] >= row['game_added_p75'])
    # === CRITERIO 3: Alto Engagement ===    
    high_engagement = (row['playtime'] >= row['playtime_p75'] and row['game_rating'] >= 3.5)
    # === CRITERIO 4: Jugadores Activos ===    
    active_community = (row['playing'] >= row['playing_p75'])

     # === DECISIÓN FINAL ===
    score = sum([high_quality, high_popularity, high_engagement, active_community]) # Éxito si cumple AL MENOS 2 de 4 criterios
    return 1 if score >= 2 else 0

df["is_success_v2"] = df.apply(calculate_success, axis=1)

pd.crosstab(df["is_success"], df["is_success_v2"], rownames=["SQL_is_success"], colnames=["metric_2of4"])


metric_2of4,0,1
SQL_is_success,,
1,101910,35269


In [ ]:
"""Existe un desbalance moderado (aprox 255 de positivos y 75% de negativos) que permite el entrenamiento con XGBoost.
El modelo tenderá a predecir más 0, la Accuracy será engañosa.
La definición de éxito (>= 2 de 4 criterios) es estricta, naturalmente reduce positivos, es coherente con el resultado.
El desbalance es estructural y justificable, no un error de datos."""

## 4) Features


In [15]:
NUMERIC_FEATURES = [
    'game_rating', 'ratings_count',
    'game_added', 'suggestions_count',
    'playtime', 'playing', 'owned', 'toplay',
    'num_platforms', 'num_stores', 'num_genres', 'num_tags'
]

CATEGORICAL_FEATURES = [
    'esrb_name', 'release_year',
    'has_multiplayer', 'has_singleplayer', 'is_indie'
]

DERIVED_FEATURES = [
    'rating_popularity_ratio',
    'engagement_score',
    'years_since_release',
    'quality_confidence',
]

TARGET = "is_success_v2"

all_features = NUMERIC_FEATURES + CATEGORICAL_FEATURES + DERIVED_FEATURES
[c for c in all_features + [TARGET] if c not in df.columns]


[]

## 5) Limpieza + recency_score


In [16]:
df_fe = df.copy()

# Tipos numéricos
for c in NUMERIC_FEATURES + DERIVED_FEATURES:
    df_fe[c] = pd.to_numeric(df_fe[c], errors="coerce")

df_fe["release_year"] = pd.to_numeric(df_fe["release_year"], errors="coerce").astype("Int64")

# flags a int
for c in ["has_multiplayer", "has_singleplayer", "is_indie"]:
    df_fe[c] = pd.to_numeric(df_fe[c], errors="coerce").fillna(0).astype(int)

df_fe = df_fe.replace([np.inf, -np.inf], np.nan)

CURRENT_YEAR = datetime.now().year
df_fe["years_since_release"] = (CURRENT_YEAR - df_fe["release_year"].astype(float))
df_fe["recency_score"] = 1.0 / (1.0 + df_fe["years_since_release"].clip(lower=0))

if "recency_score" not in DERIVED_FEATURES:
    DERIVED_FEATURES = DERIVED_FEATURES + ["recency_score"]
    all_features = NUMERIC_FEATURES + CATEGORICAL_FEATURES + DERIVED_FEATURES

df_fe[["release_year", "years_since_release", "recency_score"]].describe(include="all")


,release_year,years_since_release,recency_score
count,137179.0,137179.000000,137179.000000
mean,2020.271959,5.728041,0.191453
std,2.80769,2.807690,0.116451
min,2016.0,0.000000,0.090909
25%,2018.0,3.000000,0.111111
50%,2020.0,6.000000,0.142857
75%,2023.0,8.000000,0.250000
max,2026.0,10.000000,1.000000


## 6) Dataset final + guardado


In [17]:
keep_cols = ["game_id", "game_name"] + all_features + [TARGET]
df_model_ready = df_fe[keep_cols].dropna(subset=[TARGET])

for c in NUMERIC_FEATURES + DERIVED_FEATURES:
    df_model_ready[c] = df_model_ready[c].fillna(0)

df_model_ready["esrb_name"] = df_model_ready["esrb_name"].fillna("Unknown").astype(str)

df_model_ready.shape, df_model_ready[TARGET].value_counts(normalize=True)


((137179, 25),
 is_success_v2
 0    0.742898
 1    0.257102
 Name: proportion, dtype: float64)

In [22]:
from pathlib import Path

out_dir = Path("C:/Users/crisr/dev/rawg-aws-ml-analytics/notas_cris/00_data")
out_dir.mkdir(parents=True, exist_ok=True)

parquet_path = out_dir / "train_dataset.parquet"
df_model_ready.to_parquet(parquet_path, index=False)

parquet_path


WindowsPath('C:/Users/crisr/dev/rawg-aws-ml-analytics/notas_cris/00_data/train_dataset.parquet')

## 7) Checks rápidos


In [23]:
y = df_model_ready[TARGET]
print("Distribución target:")
print(y.value_counts())
print("\nProporción éxito:", round(y.mean(), 4))

nunique = df_model_ready[all_features].nunique(dropna=False).sort_values()
nunique.head(15)


Distribución target:
is_success_v2
0    101910
1     35269
Name: count, dtype: int64

Proporción éxito: 0.2571


has_singleplayer         2
has_multiplayer          2
is_indie                 2
game_rating              5
esrb_name                7
num_stores              10
release_year            11
num_genres              11
years_since_release     11
recency_score           11
num_platforms           12
num_tags                96
playtime               124
playing                201
toplay                 398
dtype: int64